In [1]:
# this notebook will normalize the data
# while keeping key level as feature
import pandas as pd
import json
from pathlib import Path

model_name = "BTCUSD-15m"
folder_path = Path("../../data/mlData")
src_path = folder_path / "clean-v4" / f"{model_name}-v4-cleaned-input.jsonl"

# Read JSONL file - keep timestamp as raw number
df = pd.read_json(src_path, lines=True, convert_dates=False)
df.head()

,timestamp,vol,isSTR,isITR,isLTR,time_sin,time_cos,is_mon,is_tue,is_wed,...,is_sun,c_EMA50_delta,c_EMA200_delta,close_pct_w_range,barsSinceITR,barsSinceLTR,closeDTK_LTR,closeDTK_ITR,Y,body_pct
0,1709263800000,313.675415,0,0,0,0.793353,0.608761,0,0,0,...,0,-0.007270,0.007638,0.777702,35,35,0.010585,0.010585,1,0.003486
1,1709264700000,395.454865,0,0,0,0.831470,0.555570,0,0,0,...,0,-0.005365,0.009206,0.785289,36,36,0.012279,0.012279,1,0.001676
2,1709265600000,253.236816,0,0,0,0.866025,0.500000,0,0,0,...,0,-0.004023,0.010263,0.790604,37,37,0.013466,0.013466,1,0.001172
3,1709266500000,286.789307,0,0,0,0.896873,0.442289,0,0,0,...,0,-0.001087,0.012984,0.803716,38,38,0.016394,0.016394,-1,0.002889
4,1709267400000,222.761841,0,0,0,0.923880,0.382683,0,0,0,...,0,-0.001879,0.012007,0.799771,39,39,0.015513,0.015513,1,-0.000867


In [2]:
df = df.drop(columns=["timestamp"])
print(df.shape)
df.head()

(70258, 22)


,vol,isSTR,isITR,isLTR,time_sin,time_cos,is_mon,is_tue,is_wed,is_thu,...,is_sun,c_EMA50_delta,c_EMA200_delta,close_pct_w_range,barsSinceITR,barsSinceLTR,closeDTK_LTR,closeDTK_ITR,Y,body_pct
0,313.675415,0,0,0,0.793353,0.608761,0,0,0,0,...,0,-0.007270,0.007638,0.777702,35,35,0.010585,0.010585,1,0.003486
1,395.454865,0,0,0,0.831470,0.555570,0,0,0,0,...,0,-0.005365,0.009206,0.785289,36,36,0.012279,0.012279,1,0.001676
2,253.236816,0,0,0,0.866025,0.500000,0,0,0,0,...,0,-0.004023,0.010263,0.790604,37,37,0.013466,0.013466,1,0.001172
3,286.789307,0,0,0,0.896873,0.442289,0,0,0,0,...,0,-0.001087,0.012984,0.803716,38,38,0.016394,0.016394,-1,0.002889
4,222.761841,0,0,0,0.923880,0.382683,0,0,0,0,...,0,-0.001879,0.012007,0.799771,39,39,0.015513,0.015513,1,-0.000867


# Prepare input and output
- exclude doji 26 row where Y = 0
- use 200 bars as input
- save last 500 bars as testing sessions

In [3]:
# split into train / test
# split into train / test
# Y=0 rows (26 doji artifacts) are intentionally kept for sequence continuity.
# They will be excluded as prediction targets during sequence building in training.
test_df  = df.iloc[-500:].copy()
train_df = df.iloc[:-500].copy()

print(f"Train: {train_df.shape}, Test: {test_df.shape}")
print(f"Train Y=0: {(train_df['Y'] == 0).sum()}, Test Y=0: {(test_df['Y'] == 0).sum()}")



Train: (69758, 22), Test: (500, 22)
Train Y=0: 26, Test Y=0: 0


In [4]:
# apply scaler
from sklearn.preprocessing import StandardScaler
import joblib
from datetime import date

formatted_date = date.today().strftime("%Y%m")

# Continuous features only — isSTR/isITR/isLTR are ternary ordinal, skip scaling
scale_cols = [
    "vol", "body_pct",
    "c_EMA50_delta", "c_EMA200_delta", "close_pct_w_range",
    "barsSinceITR", "barsSinceLTR",
    "closeDTK_LTR", "closeDTK_ITR",
]

scaler = StandardScaler()
train_df[scale_cols] = scaler.fit_transform(train_df[scale_cols])  # fit + transform on train only
test_df[scale_cols]  = scaler.transform(test_df[scale_cols])       # transform only

joblib.dump(scaler, folder_path / "scaler" / f"{formatted_date}-{model_name}-scaler-v4.pkl")
print("Scaler saved. Train:", train_df.shape, "Test:", test_df.shape)


Scaler saved. Train: (69758, 22) Test: (500, 22)


In [5]:
# save train data
# where Y = 0 around 26 cells, remove them during training
train_path = folder_path / "trainData" / f"{formatted_date}-{model_name}-train-v4.jsonl"
test_path  = folder_path / "trainData" / f"{formatted_date}-{model_name}-test-v4.jsonl"

train_path.parent.mkdir(parents=True, exist_ok=True)

train_df.to_json(train_path, orient="records", lines=True)
test_df.to_json(test_path,  orient="records", lines=True)

print(f"Saved train: {train_path.name}")
print(f"Saved test:  {test_path.name}")


Saved train: 202603-BTCUSD-15m-train-v4.jsonl
Saved test:  202603-BTCUSD-15m-test-v4.jsonl
